# 26장 · LangGraph StateGraph의 상태와 노드

StateGraph의 노드는 현재 상태를 읽고 바꿀 필드를 반환합니다. 간선은 그 변경을 반영한 뒤 어느 노드를 실행할지 정합니다.

질문 정리·문서 검색·답변 작성을 StateGraph의 세 노드로 연결합니다. 각 함수의 반환값과 갱신된 전체 상태를 비교합니다. Python 커널에 LangGraph와 langchain-core가 필요하며 검색과 답변은 로컬 규칙을 사용합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## TypedDict와 노드의 부분 갱신

State는 한 질문을 처리하면서 여러 단계가 함께 읽는 값입니다. 여기서는 question에 질문, documents에 찾은 문장 목록, answer에 최종 안내를 담습니다. 노드가 값을 읽어 바꿀 부분만 돌려주면 그래프가 앞 상태에 합칩니다.

### 노드 실행 순서와 공유 상태

<p>Node는 이번에 실행할 Python 함수입니다. Edge는 어떤 함수 다음에 어떤 함수를 실행할지 정한 연결입니다. State는 실행 중 공유하는 값의 묶음입니다. 문서가 어디에 저장되는지와 어느 함수가 다음에 호출되는지를 한 용어로 섞으면 오류 원인을 찾기 어렵습니다.</p>

<p>예를 들어 질문의 공백을 지우는 normalize는 question을 읽고 question만 갱신합니다. 문서를 고르는 lookup은 question을 읽고 documents를 갱신합니다. 답변을 만드는 compose는 documents를 읽고 answer를 갱신합니다. 이번 그래프에서는 모델이 다음 노드를 고르지 않고 코드의 고정된 연결을 따릅니다.</p>

<table><caption>세 요소를 코드에 대응하기</caption><thead><tr><th scope="col">요소</th><th scope="col">예제</th><th scope="col">변경의 의미</th></tr>
</thead><tbody><tr><td>Node</td><td>normalize, lookup, compose</td><td>처리할 계산을 바꿈</td></tr>
<tr><td>Edge</td><td>normalize → lookup</td><td>실행 순서를 바꿈</td></tr>
<tr><td>State</td><td>question, documents, answer</td><td>전달하고 보관할 값을 바꿈</td></tr>
</tbody></table>

### 셀 01 · normalize()의 공백 정리

초기 상태를 normalize에 직접 전달해 반환된 필드를 확인합니다.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    documents: list[str]
    answer: str

def normalize(state: State) -> dict:
    question = state["question"].strip()
    if not question:
        raise ValueError("질문을 입력하세요.")
    return {"question": question}

initial = {'question': '  대출 기간은?  ', 'documents': [], 'answer': ''}
print(normalize(initial))

{'question': '대출 기간은?'}


strip()이 앞뒤 공백을 제거해 {"question": "대출 기간은?"}을 반환했습니다.

normalize(initial)을 직접 호출한 줄에서 결과의 documents를 읽으려고 하면 키가 없습니다. 반환한 것은 이 노드가 바꾼 항목 하나뿐입니다. 그래프에서는 그 항목을 기존 상태에 반영하므로 다음 노드는 정리된 질문과 처음의 빈 문서 목록을 함께 받습니다.

<div class="example-table"><table><caption>normalize 호출 전후의 필드</caption><thead><tr><th scope="col">관찰 위치</th><th scope="col">question</th><th scope="col">documents</th><th scope="col">answer</th></tr></thead><tbody><tr><td>초기 상태</td><td>"  대출 기간은?  "</td><td>[]</td><td>""</td></tr><tr><td>normalize의 반환값</td><td>"대출 기간은?"</td><td>반환 생략</td><td>반환 생략</td></tr><tr><td>변경을 반영한 상태</td><td>"대출 기간은?"</td><td>[]</td><td>""</td></tr></tbody></table></div>

### TypedDict 상태 스키마

<p>State는 실행 중 사용할 딕셔너리의 키와 값 형태를 적은 타입입니다. question은 질문 문자열, documents는 문장 문자열의 리스트, answer는 답변 문자열입니다. 실제 값은 graph.invoke에 넘기는 initial에 들어 있습니다. 타입 선언만으로 질문이 자동 생성되지는 않습니다.</p>

<p>TypedDict는 정적 타입 확인을 돕는 선언입니다. 잘못된 자료형이나 빈 입력을 모든 경우 자동으로 막아 주는 런타임 검증기로 생각하지 마세요. 이번 예제는 question.strip() 결과가 비었는지 노드에서 직접 검사합니다.</p>

<table><caption>초기 상태 예측</caption><thead><tr><th scope="col">키</th><th scope="col">초기 값</th><th scope="col">누가 갱신하나요?</th></tr>
</thead><tbody><tr><td>question</td><td>"  대출 기간은?  "</td><td>normalize</td></tr>
<tr><td>documents</td><td>[]</td><td>lookup</td></tr>
<tr><td>answer</td><td>""</td><td>compose</td></tr>
</tbody></table>

### 셀 02 · 노드 반환값과 상태 병합

직접 호출에서는 딕셔너리를 합쳐 그래프가 하는 일을 한 번 따라 해 봅니다.

In [2]:
normalized = {**initial, **normalize(initial)}
print(normalized)

{'question': '대출 기간은?', 'documents': [], 'answer': ''}


documents와 answer의 빈 값이 남습니다. 두 필드는 이제 검색과 답변 노드가 채울 자리입니다.

#### 노드 하나의 반환값과 그래프의 상태

<mark>normalize가 question만 반환해도 기존 documents와 answer는 상태에 남습니다.</mark> 직접 호출한 결과와 다음 셀의 병합 결과를 나란히 보세요. 그래프에서는 간선을 따라 다음 노드가 실행될 때 이미 갱신한 질문을 읽습니다. compile()은 연결을 준비하고 첫 실행은 invoke(initial)에서 시작합니다.

선수 개념 · 24장 LCEL로 구성하는 RAG와 인용 검증

### 노드 함수의 부분 갱신

<p>normalize가 반환한 딕셔너리에는 question 하나만 있습니다. 반환에서 빠진 documents와 answer를 지우라는 뜻이 아닙니다. 이번처럼 별도 갱신 규칙을 지정하지 않은 필드는 해당 키가 반환되면 새 값으로 바뀌며, 반환하지 않은 필드는 유지됩니다.</p>

<p>lookup은 documents만, compose는 answer만 반환합니다. 양쪽이 입력 상태를 직접 바꾸지 않으므로 각 함수를 따로 호출해 어떤 필드를 썼는지 확인할 수 있습니다.</p>

## add_node()·add_edge()·compile()로 그래프 연결

엣지는 어떤 함수를 다음에 부를지 정하는 연결입니다. START에서 normalize, lookup, compose 순으로 연결한 뒤 END에서 멈춥니다. 검색 결과를 미리 써 넣지 않고 질문에 들어 있는 낱말을 자료와 대조합니다.

### 셀 03 · StateGraph의 노드와 간선

이번 셀의 lookup은 노트북 안에 넣어 둔 두 가상 문장에서 찾습니다. 등록된 노드 이름과 엣지의 이름을 맞춰 보세요.

초기값부터 normalize, lookup, compose 순서로 따라가세요. 한 노드가 돌려주는 사전은 이번에 바꿀 칸만 담습니다. 나머지 두 칸은 앞 상태의 값을 유지합니다.

In [3]:
rows = [{'id': 'library-1', 'keyword': '대출', 'text': '도서 대출 기간은 14일입니다.'},
 {'id': 'library-2', 'keyword': '휴관', 'text': '도서관은 월요일에 휴관합니다.'}]

def lookup(state):
    found = [r['text'] for r in rows if r['keyword'] in state['question']]
    return {'documents': found}

def compose(state: State) -> dict:
    documents = state["documents"]
    answer = " / ".join(documents) if documents else "제공된 문서에서 확인할 수 없습니다."
    return {"answer": answer}

def make_graph():
    builder = StateGraph(State)
    builder.add_node("normalize", normalize)
    builder.add_node("lookup", lookup)
    builder.add_node("compose", compose)
    builder.add_edge(START, "normalize")
    builder.add_edge("normalize", "lookup")
    builder.add_edge("lookup", "compose")
    builder.add_edge("compose", END)
    return builder.compile()

graph = make_graph()
print(graph.invoke(initial)['answer'])

도서 대출 기간은 14일입니다.


14일이라는 문장이 나옵니다. lookup이 대출 문장을 골라 documents에 넣었고 compose는 그대로 이어 붙였습니다. 모델에 날짜를 묻는 과정은 없습니다.

#### invoke 한 번에서 바뀌는 상태 필드

```mermaid
flowchart TD
s["START"] --> n["normalize · 질문 공백 정리"]
n --> l["lookup · 문서 검색"]
l --> c["compose · 답변 작성"]
c --> e["END"]
```

normalize는 question, lookup은 documents, compose는 answer를 반환합니다. 앞 노드가 써 둔 나머지 필드도 최종 상태에 남습니다.

### 노드·간선 등록과 컴파일

<p>StateGraph는 그래프를 만드는 빌더입니다. add_node의 첫 인자는 그래프에서 사용하는 이름이고 두 번째 인자는 실행할 함수입니다. add_edge는 등록한 이름으로 연결합니다. START는 입력을 받은 시작 위치, END는 종료 위치입니다.</p>

<p>compile()은 등록된 노드와 간선을 실행 가능한 그래프로 묶습니다. 이때 lookup의 본문은 아직 실행되지 않습니다. 어느 문장을 골랐는지는 invoke() 뒤의 documents에서 확인합니다.</p>

### 같은 상태에서 세 노드가 한 칸씩 채웁니다

실행 중인 노드가 바뀔 때 반환 키도 바뀝니다. lookup이 찾은 문장은 compose가 answer를 쓴 뒤에도 documents에 남습니다.

1. START · question = "  대출 기간은?  "
2. normalize · question = "대출 기간은?"
3. lookup · documents[0] = 도서 대출 기간은 14일입니다.
4. compose · answer = 도서 대출 기간은 14일입니다.
5. END · 최종 상태 = question · documents · answer

### 과제 04 · invoke() 결과의 근거 확인

질문을 ‘휴관 요일은?’로 바꾸어 실행하고 최종 상태에서 documents만 표시하세요.

In [4]:
# 초기 상태의 질문을 바꾼 뒤 그래프에 넘기세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>request = {**initial, &#x27;question&#x27;: &#x27;휴관 요일은?&#x27;}
print(graph.invoke(request)[&#x27;documents&#x27;])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;도서관은 월요일에 휴관합니다.&#x27;]</code></pre>
<p>휴관에 해당하는 문장 한 개가 목록에 남습니다. answer가 문자열인 데 비해 이 칸은 검색이 여러 건일 수 있어 목록입니다.</p>
</details>

## 돌아보기

주차 질문을 넣었을 때 답변을 만든 노드와 빈 목록을 만든 노드를 구별해 보세요. 다음 장은 빈 검색 뒤 한 번 더 찾을지 결정합니다.

### 과제 05 · 근거 두 줄을 쓰는 답변 노드

대출 기간과 휴관 요일을 함께 물었습니다. 기존 lookup의 결과를 줄 단위 답변으로 바꾸는 compose_lines 노드를 직접 작성하고 그래프에 등록하세요. 최종 상태에서 문서 수와 답변을 읽습니다.

In [ ]:
question = "대출 기간과 휴관 요일은?"
# compose_lines는 answer만 반환합니다. normalize → lookup → compose_lines로 연결하세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>question = "대출 기간과 휴관 요일은?"

def compose_lines(state):
    docs = state["documents"]
    return {"answer": "\n".join(docs) if docs else "근거 없음"}

course = StateGraph(State)
course.add_node("normalize", normalize)
course.add_node("lookup", lookup)
course.add_node("compose_lines", compose_lines)
course.add_edge(START, "normalize")
course.add_edge("normalize", "lookup")
course.add_edge("lookup", "compose_lines")
course.add_edge("compose_lines", END)
result = course.compile().invoke({**initial, "question": question})
print(len(result["documents"]))
print(result["answer"])</code></pre>
<p>실행 결과</p><pre><code>2
도서 대출 기간은 14일입니다.
도서관은 월요일에 휴관합니다.</code></pre>
<p>마지막 노드가 돌려준 딕셔너리는 answer 한 칸입니다. 최종 상태에는 lookup이 찾은 documents 두 건도 남으므로 실제로 어떤 자료를 요약했는지 대조할 수 있습니다.</p>
</details>

## 문서 수를 세는 노드를 그래프에 끼워 넣기

앞에서 lookup은 문서 목록을, compose는 답변을 만들었습니다. 검색과 답변 사이에 “근거 몇 건을 읽었는가”라는 값을 남기되 이미 동작하는 두 함수를 재사용합니다.

### 셀 06 · 노드별 반환값에 남은 필드 읽기

대출과 휴관을 함께 묻습니다. stream의 updates에서 노드가 이번 차례에 쓴 필드 이름만 꺼내 보세요.

In [5]:
both = {**initial, "question": "대출 기간과 휴관일은?"}
for update in graph.stream(both, stream_mode="updates"):
    name, delta = next(iter(update.items()))
    print(name, list(delta))

normalize ['question']
lookup ['documents']
compose ['answer']


normalize는 question, lookup은 documents, compose는 answer만 씁니다. 어떤 단계도 문서 수를 따로 저장하지 않았습니다. 마지막 단계만 보면 문서가 몇 건 있었는지 알려면 다시 목록을 세어야 합니다.

### 과제 07 · 기존 노드 사이에 evidence_count 기록하기

State를 상속해 evidence_count 정수 필드를 추가하세요. lookup 직후에 count 노드를 두고, 같은 질문에 대한 최종 개수와 답변을 출력하세요.

In [6]:
# class AuditedState(State)와 count 노드를 정의한 뒤 기존 함수들을 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>class AuditedState(State):
    evidence_count: int

def count_evidence(state):
    return {&quot;evidence_count&quot;: len(state[&quot;documents&quot;])}

audit_builder = StateGraph(AuditedState)
audit_builder.add_sequence([(&quot;normalize&quot;, normalize), (&quot;lookup&quot;, lookup),
                           (&quot;count&quot;, count_evidence), (&quot;compose&quot;, compose)])
audit_builder.add_edge(START, &quot;normalize&quot;)
audit_builder.add_edge(&quot;compose&quot;, END)
audit_graph = audit_builder.compile()
audit_result = audit_graph.invoke({**both, &quot;evidence_count&quot;: 0})
print(audit_result[&quot;evidence_count&quot;], audit_result[&quot;answer&quot;])</code></pre>
<p>실행 결과</p><pre><code>2 도서 대출 기간은 14일입니다. / 도서관은 월요일에 휴관합니다.</code></pre>
<p>값은 2이고 두 안내 문장이 이어집니다. count는 documents를 다시 반환하지 않았지만 앞 노드의 문서가 상태에 남아 compose도 읽을 수 있습니다. count를 lookup 앞에 두면 처음 넣었던 빈 목록을 세게 됩니다.</p>
</details>

<mark>노드가 읽을 수 있는 값은 연결 순서에 따라 달라집니다.</mark> count의 코드를 그대로 두고 실행 위치만 앞당겨도 기록되는 숫자가 달라집니다.

27장은 노드의 실행 순서를 상태값이 바꾸는 경우를 다룹니다.